# Task 3 – Merge and quality checks
**Who:** Joud & Maram  
**Input:** `food_clean.csv` (Task2_Cleaning), `joud_features.csv` (Task2_Joud_Violations), `maram_features.csv` (Task2_Maram_History). Run those three notebooks first, on the same computer.  
**Output:** `food_final.csv`, the final dataset for the model.

## 1. Merge the three files

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pandas for the tables and matplotlib for the box plots in the outlier check.

In [ ]:
clean = pd.read_csv('food_clean.csv')
print(clean.shape)

The clean file: one row per real inspection, with the original columns plus Business_ID, First_Seen_Date, Year, Fail and Period. (Write here the shape.)

In [ ]:
joud = pd.read_csv('joud_features.csv')
print(joud.shape)

Joud's violation features (counts, severity levels, topics, previous-visit columns). (Write here the shape.)

In [ ]:
maram = pd.read_csv('maram_features.csv')
print(maram.shape)

Maram's history and calendar features. (Write here the shape.)

In [ ]:
print(clean['Inspection ID'].duplicated().sum(), joud['Inspection ID'].duplicated().sum(), maram['Inspection ID'].duplicated().sum())

Before merging we check that `Inspection ID` is unique in each file, so every inspection appears only once. All three numbers should be 0.

In [ ]:
final = clean.merge(joud, on='Inspection ID', how='inner', validate='one_to_one')
final = final.merge(maram, on='Inspection ID', how='inner', validate='one_to_one')
print(final.shape)

We join the three files on `Inspection ID` (`merge` is new for us: it puts the columns of two tables side by side, matching rows with the same ID, like VLOOKUP in a spreadsheet). `validate='one_to_one'` stops with an error if an ID appears twice. The number of rows must stay the same as in the clean file. (Write here the shape.)

## 2. Quality checks

In [ ]:
print(len(final) == len(clean) == len(joud) == len(maram))

Quality check 1: no rows were lost or added by the merge (should print True).

In [ ]:
final.isnull().sum()[final.isnull().sum() > 0]

Quality check 2: the only columns with missing values should be descriptive columns we do not use (AKA Name, City, State, Zip, Latitude, Longitude, Location) and Violations (empty = no violations). All feature columns must have 0 missing values. (Write here what you see.)

In [ ]:
final['Inspection Date'] = pd.to_datetime(final['Inspection Date'])
print(final['Inspection Date'].min(), final['Inspection Date'].max())

Quality check 3: the dates are between 1 January 2015 and 31 December 2022.

In [ ]:
final['Results'].value_counts()

Quality check 4: only the three real results are left.

### Quality check 5 – Outliers
We check the numeric columns in two steps: first **impossible values** (errors), then **outliers** with the IQR method from Lab 03.

In [ ]:
num_cols = ['Num_Violations', 'Num_Critical', 'Num_Serious', 'Prev_Num_Violations',
            'Days_Since_Last', 'Days_Since_First_Seen', 'Num_Past_Inspections']
(final[num_cols] < 0).sum()

Impossible values: a count of violations or a number of days can never be negative. (Write here: all columns should show 0. If one is not 0, it is an error and must be fixed.)

In [ ]:
final[num_cols].describe().round(1)

Summary of the numeric columns: we compare the maximum with the 75% value to see which columns have very large values. (Write here the columns where the max is much bigger than the 75% value.)

In [ ]:
for col in num_cols:
    q1 = final[col].quantile(0.25)
    q3 = final[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - (1.5 * iqr)
    upper_fence = q3 + (1.5 * iqr)
    n_out = ((final[col] < lower_fence) | (final[col] > upper_fence)).sum()
    print(f'{col}: fences {lower_fence:.1f} to {upper_fence:.1f}, outliers = {n_out} ({n_out / len(final) * 100:.1f}%)')

IQR method (same as Lab 03): a value below Q1 − 1.5·IQR or above Q3 + 1.5·IQR is an outlier. (Write here the number and % of outliers in each column.)

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot([final['Num_Violations'], final['Num_Critical'], final['Num_Serious']])
plt.xticks([1, 2, 3], ['Num_Violations', 'Num_Critical', 'Num_Serious'])
plt.title('Box Plot of the Violation Counts')
plt.xlabel('Column')
plt.ylabel('Number of violations per inspection')
plt.show()

Box plot of the violation counts. The points above the whiskers are the outliers. (Write here what you see.)

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot([final['Days_Since_Last'], final['Days_Since_First_Seen']])
plt.xticks([1, 2], ['Days_Since_Last', 'Days_Since_First_Seen'])
plt.title('Box Plot of the Time Columns')
plt.xlabel('Column')
plt.ylabel('Days')
plt.show()

Box plot of the time columns (in days, so in a separate plot because the scale is very different). (Write here what you see.)

In [ ]:
q1 = final['Num_Violations'].quantile(0.25)
q3 = final['Num_Violations'].quantile(0.75)
upper_fence = q3 + (1.5 * (q3 - q1))
is_outlier = final['Num_Violations'] > upper_fence
is_outlier.groupby(final['Period']).mean().round(3)

Share of inspections with an outlier number of violations in each period. (Write here. In Task 1 we saw that long violation lists are much more common after the change, so we expect a much bigger share in Post than in Pre.)

**Decision – we keep the outliers.**
- They are **real inspections**, not typing errors: there are no impossible (negative) values, and a business can really get many violations or go a long time without an inspection.
- The long violation lists are **part of what we study**: they became much more common after the 2018 change. Removing them would hide the effect we test in H1b.
- Our test for the number of violations (**Mann–Whitney U**) uses ranks, so outliers do not distort it.
- For the model (Phase 2), decision trees are not sensitive to outliers; if we use logistic regression we can cap the extreme values then.

In [ ]:
final['Fail'].mean().round(3)

The share of failed inspections in the final dataset (the target). (Write here the share. About 1 in 5 inspections fails, so the classes are unbalanced; this matters for the model.)

In [ ]:
final['Is_Reinspection'] = final['Is_Reinspection'].astype(int)
final.to_csv('food_final.csv', index=False)
print(final.shape)

We save the final dataset. `Is_Reinspection` is stored as 0/1 like the other yes/no columns. The meaning of every column is in `README.md`.